# 02 - Pipeline validation
Row counts, expectations and sample gold output from the Lakeflow declarative pipeline.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../scripts'))
import pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
from dbx import sql, ws, CATALOG
print('workspace:', ws().config.host, '| catalog:', CATALOG, '| user:', ws().current_user.me().user_name)

workspace: https://fevm-serverless-stable-oj7eie.cloud.databricks.com | catalog: serverless_stable_oj7eie_catalog | user: erica.yuen@databricks.com


In [2]:
from export_evidence import pipeline_id
pid, name = pipeline_id(); print(pid, name)
ups = ws().pipelines.list_updates(pid, max_results=5).updates
pd.DataFrame([{'update_id': u.update_id, 'state': u.state.value, 'full_refresh': u.full_refresh} for u in ups])

41d741ff-7755-44b7-9cfc-bb305a327764 [dev erica_yuen] matter-intelligence-pipeline


,update_id,state,full_refresh
0,bd2a970d-5353-4f19-b869-45baa764f83a,COMPLETED,False
1,3f14260a-357f-4eb1-b033-ef7e21bd81f9,COMPLETED,True
2,c4c6a62f-7df9-4616-af10-8b2e25c8b519,COMPLETED,False


In [3]:
from export_evidence import TABLES
sql(' UNION ALL '.join("SELECT '%s' t, count(*) n FROM ${catalog}.%s" % (t, t) for t in TABLES))

,t,n
0,curated.gold_documents,20
1,curated.gold_clauses,166
2,curated.gold_risk_flags,63
3,raw.bronze_documents,20
4,curated.silver_parsed,20
5,curated.int_doc_classified,20
6,curated.int_doc_extracted,20
7,curated.int_clauses,166
8,curated.int_risk_flags,63
9,curated.gold_cost_metrics,5


In [4]:
sql(f"""WITH ev AS (SELECT origin.update_id, explode(from_json(details:flow_progress.data_quality.expectations::string,
 'array<struct<name:string,dataset:string,passed_records:bigint,failed_records:bigint>>')) e
 FROM event_log('{pid}') WHERE event_type='flow_progress' AND details:flow_progress.data_quality.expectations IS NOT NULL)
SELECT update_id, e.dataset, e.name, sum(e.passed_records) passed, sum(e.failed_records) failed FROM ev GROUP BY ALL ORDER BY 1,2,3""")

,update_id,dataset,name,passed,failed
0,3f14260a-357f-4eb1-b033-ef7e21bd81f9,serverless_stable_oj7eie_catalog.curated.int_clauses,confidence_not_null,166,0
1,3f14260a-357f-4eb1-b033-ef7e21bd81f9,serverless_stable_oj7eie_catalog.curated.silver_parsed,parse_ok,20,0
2,3f14260a-357f-4eb1-b033-ef7e21bd81f9,serverless_stable_oj7eie_catalog.curated.silver_parsed,text_min_length,20,0
3,3f14260a-357f-4eb1-b033-ef7e21bd81f9,serverless_stable_oj7eie_catalog.raw.bronze_documents,has_content,20,0
4,c4c6a62f-7df9-4616-af10-8b2e25c8b519,serverless_stable_oj7eie_catalog.curated.int_clauses,confidence_not_null,167,0
5,c4c6a62f-7df9-4616-af10-8b2e25c8b519,serverless_stable_oj7eie_catalog.curated.silver_parsed,parse_ok,20,0
6,c4c6a62f-7df9-4616-af10-8b2e25c8b519,serverless_stable_oj7eie_catalog.curated.silver_parsed,text_min_length,20,0
7,c4c6a62f-7df9-4616-af10-8b2e25c8b519,serverless_stable_oj7eie_catalog.raw.bronze_documents,has_content,20,0


In [5]:
sql("SELECT doc_id, page_count, parse_status, length(full_text) chars, parse_error FROM ${catalog}.curated.silver_parsed ORDER BY doc_id")

,doc_id,page_count,parse_status,chars,parse_error
0,007493cb2c881bc464f4ee102044914082f89a8018fced6482358426d9bb7340,10,ok,34812,None
1,389c45485e279c61849d74904e6b458754dcb546cf328e9ba4c3289cf4615b96,25,ok,54453,None
2,4fa0cc0e911869eaece3a9f0e74b853adbab23f213ad711296228527d13e8f85,12,ok,28254,None
3,4fd6f7a48150f89bd6dde6171dfe034e189599708a248360357d77756dd1309f,13,ok,72361,None
4,61e3fcfe8b24f6ed4911645c1dc6cc9338d22fe080aae28801393fb32f9f53c2,7,ok,13348,None
5,661d35cc01bcb542a595c245b317e8a4e508dfa62c32cee0125afae43d5d08f2,8,ok,15696,None
6,69c81aa59f1b17d3c8df7864b29cd6070ed5ebfafc30d55b17ceb883c7ad7563,19,ok,29991,None
7,6d000e8f8d38b738a040dc450e96d8708ea944dc56db5fce5472f974a6e9144a,12,ok,28017,None
8,8dc7e7725a0ad10d04aa66b8cdb279a395d0211a8c4720361baf012f3bf3bcb6,9,ok,28434,None
9,946b7c3b83cc99907b75617a6192816788551613df6edb5c0404dd8a0fe0c5c6,71,ok,229808,None


In [6]:
sql("SELECT contract_type, count(*) docs, round(avg(contract_type_confidence),3) conf FROM ${catalog}.curated.gold_documents GROUP BY 1 ORDER BY 2 DESC")

,contract_type,docs,conf
0,Other,6,0.71
1,Distribution,6,0.79
2,License,4,0.835
3,MSA,3,0.78
4,Supply,1,0.78


In [7]:
sql("SELECT clause_type, count(*) n, round(avg(confidence),3) conf FROM ${catalog}.curated.gold_clauses GROUP BY 1 ORDER BY 1")

,clause_type,n,conf
0,assignment_restriction,16,0.997
1,auto_renewal,15,0.94
2,effective_date,18,0.994
3,exclusivity,10,0.955
4,expiration_date,12,0.917
5,governing_law,17,0.994
6,indemnification,12,0.962
7,liability_cap,8,0.931
8,non_compete,8,0.981
9,notice_period_days,5,0.88


In [8]:
sql("SELECT clause_type, normalized_value, round(confidence,2) conf, left(source_snippet, 150) snippet FROM ${catalog}.curated.gold_clauses WHERE clause_type IN ('governing_law','auto_renewal','uncapped_liability','notice_period_days') ORDER BY doc_id, clause_type LIMIT 12")

,clause_type,normalized_value,conf,snippet
0,auto_renewal,true,1.0,with this Agreement. This Agreement shall automatically extend for an additional two (2) years on the same terms her...
1,governing_law,California,1.0,g. Governing Law and Forum. This Agreement will for all purposes be governed by and interpreted in accordance with t...
2,uncapped_liability,true,1.0,"c. LIMITATIONS ON LIABILITY/NO INJUNCTIVE RELIEF. EXCEPT IN CASES OF GROSS NEGLIGENCE, WILLFUL\nMISCONDUCT OR FRAUD,..."
3,auto_renewal,false,1.0,"4.1 <u>Term.</u> The term of this Agreement (the “<u>Term”),</u> unless mutually extended by written agreement of th..."
4,governing_law,Ohio,1.0,6.5 <u>Governing Law; Venue.</u> This Agreement shall be governed by and construed in accordance with the laws of th...
5,auto_renewal,false,1.0,NaN
6,governing_law,Connecticut,1.0,This Agreement shall be governed by and construed in accordance with the laws of the State of Connecticut.\n18. RELA...
7,uncapped_liability,true,1.0,20. INDEMNITY AND INSURANCE. Berkshire shall indemnify and hold Auriemma harmless from and against any and all claim...
8,auto_renewal,false,1.0,NaN
9,governing_law,Washington,1.0,"11.11 Choice of Law. This Agreement will be interpreted, construed and\nenforced in accordance with the Laws of the ..."


In [9]:
sql("SELECT severity, rule_id, count(*) flags, count(explanation) explained FROM ${catalog}.curated.gold_risk_flags GROUP BY ALL ORDER BY 1,2")

,severity,rule_id,flags,explained
0,high,R01,6,6
1,high,R02,8,8
2,low,R06,11,0
3,low,R07,6,0
4,low,R08,16,0
5,medium,R03,4,4
6,medium,R04,2,2
7,medium,R05,10,10


In [10]:
sql("SELECT rule_id, severity, left(explanation, 300) explanation FROM ${catalog}.curated.gold_risk_flags WHERE explanation IS NOT NULL LIMIT 5")

,rule_id,severity,explanation
0,R02,high,This clause triggers the non-compete rule because it explicitly restricts Allied from endorsing or permitting market...
1,R05,medium,This clause triggers the exclusivity rule because it explicitly prohibits WPT and its affiliates from granting licen...
2,R05,medium,"This clause triggers the exclusivity rule because ACSI is explicitly prohibited from offering, selling, or authorizi..."
3,R02,high,"This clause triggers the non-compete rule because it explicitly restricts the Franchisee from owning, managing, or o..."
4,R02,high,"This clause triggers the rule because it explicitly restricts each Seller from ""directly, or indirectly...own, manag..."


In [11]:
sql("SELECT * FROM ${catalog}.curated.gold_cost_metrics ORDER BY run_id, stage")

,run_id,stage,docs,pages,input_tokens,output_tokens,est_cost_usd,cost_per_1k_docs
0,20261117,classify,20,NaN,91793,600,0.0468,2.34
1,20261117,explain,13,NaN,6320,3367,0.0232,1.16
2,20261117,extract,20,NaN,212067,13912,0.1269,6.35
3,20261117,parse,20,438,NaN,NaN,2.19,109.5
4,20261117,total,20,438,310180,17879,2.3869,119.34
